# Exploratory false-positive test on MRI slices without tumor

The 10 models of the patient-level 5-fold cross-validation (adopted model and YOLOv12n, one model per test
fold, seed 42) are applied, unchanged, to the **"no tumor"** images of the public 4-class Kaggle brain MRI
dataset (Nickparvar). No training, no tuning: every detection on these images is a false positive.

**Exploratory only:** these images come from other centres (Br35H collection), mix MRI sequences and planes,
and differ from the T1-weighted contrast-enhanced Figshare slices used for training. The measured false-positive
rate therefore mixes the behaviour of the detector on normal anatomy with a domain shift.

**Kaggle settings:** Accelerator **None** (CPU, no GPU quota), Internet **on** (Ultralytics install).
Inputs: the 4-class dataset `masoudnickparvar/brain-tumor-mri-dataset`, and the private datasets
`cv5fold-runs-adopted-seed42` and `cv5fold-runs-yolov12n-seed42` (trained models). Secrets `KAGGLE_USERNAME` and
`KAGGLE_KEY` attached (used only by the progress heartbeat).
Run with *Save Version → Save & Run All (Commit)* (about 1 h); results in the Output tab.

## 0. Progress heartbeat (dataset notumor-progress, every 5 min)

In [ ]:
# Progress heartbeat: every 5 min, a tiny version of the private dataset notumor-progress with what
# the notebook is doing (current line, images processed).
# Needs the secrets KAGGLE_USERNAME and KAGGLE_KEY attached to this notebook (Add-ons > Secrets).
import json, os, socket, subprocess, sys, threading, time, traceback
from pathlib import Path

socket.setdefaulttimeout(120)  # a stalled download fails after 2 min instead of hanging the session

from kaggle_secrets import UserSecretsClient
_hb_secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = _hb_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = _hb_secrets.get_secret("KAGGLE_KEY")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.2"])

HB_ID = f"{os.environ['KAGGLE_USERNAME']}/notumor-progress"
HB_STAGE = Path("/tmp/heartbeat_stage_notumor")
HB_T0 = time.time()
HB_MAIN = threading.main_thread().ident


def _hb_tail(path, n=3):
    try:
        text = path.read_bytes()[-20000:].decode(errors="replace")
    except OSError:
        return []
    lines = [l.split("\r")[-1].strip() for l in text.split("\n")]  # last state of progress bars
    return [l[:220] for l in lines if l][-n:]


def _hb_report():
    frame = sys._current_frames().get(HB_MAIN)
    stack = traceback.extract_stack(frame) if frame is not None else []
    cell = [f for f in stack if "ipykernel" in f.filename or f.filename.startswith("<ipython")]
    where = f"cell line {cell[-1].lineno}: {cell[-1].line}" if cell else "idle (between cells or finished)"
    hours = (time.time() - HB_T0) / 3600
    lines = [f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {hours:.2f} h since the heartbeat started",
             f"notebook is at: {where}", "", "stack of the notebook (last 15 frames):"]
    lines += [f"  {Path(f.filename).name}:{f.lineno} {f.name} | {(f.line or '')[:150]}" for f in stack[-15:]]
    import __main__
    ns = vars(__main__)
    running = []
    if "neg" in ns:
        running.append(f"{len(ns['neg'])} unique images")
    if "img_rows" in ns:
        done = len(ns["img_rows"])
        per_model = len(ns["neg"]) if "neg" in ns else 0
        running.append(f"{done} predictions" + (f" ({done / per_model:.1f} of 10 models)" if per_model else ""))
    lines += ["", "progress: " + (", ".join(running) or "setup")]
    try:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().replace("\n", " ; ")
    except Exception as e:
        gpu = f"n/a ({e.__class__.__name__})"
    lines += ["", f"GPU (utilization, memory): {gpu}"]
    state = ", ".join(running) if running else ((cell[-1].line or "").strip()[:70] if cell else "idle")
    return "\n".join(lines), f"{hours:.2f} h | {state} | GPU {gpu}"[:190]


def _hb_upload(text, note):
    HB_STAGE.mkdir(parents=True, exist_ok=True)
    (HB_STAGE / "progress.txt").write_text(text)
    (HB_STAGE / "dataset-metadata.json").write_text(json.dumps(
        {"title": "notumor-progress", "id": HB_ID, "licenses": [{"name": "CC0-1.0"}]}))
    r = subprocess.run(["kaggle", "datasets", "status", HB_ID], capture_output=True, text=True, timeout=120)
    low = (r.stdout + r.stderr).lower()
    exists = r.returncode == 0 and not any(s in low for s in ("404", "not found", "forbidden"))
    args = ["version", "-p", str(HB_STAGE), "-m", note] if exists else ["create", "-p", str(HB_STAGE)]
    subprocess.run(["kaggle", "datasets", *args, "-q"], capture_output=True, text=True, timeout=600)


def _hb_loop():
    while True:
        try:
            _hb_upload(*_hb_report())
        except Exception:
            pass  # the heartbeat must never disturb the notebook
        time.sleep(300)


threading.Thread(target=_hb_loop, daemon=True, name="heartbeat").start()
print("Heartbeat started: follow the session in the dataset", HB_ID)

## 1. Configuration

In [ ]:
CONFIGS = ["adopted", "yolov12n"]
FOLDS = [1, 2, 3, 4, 5]
SEED = 42
THRESHOLDS = [0.25, 0.50]   # fixed confidence thresholds (0.25 = Ultralytics/deployment default)
IMGSZ = 640
ULTRALYTICS_VERSION = "8.4.104"
N_EXAMPLES = 8              # highest-scoring false positives saved as annotated images (fold-1 models)
CHUNK = 16                  # images per predict() call (a list is processed as ONE batch by Ultralytics)

from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

## 2. Environment

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])

import hashlib, json, os, platform, time
import numpy as np, pandas as pd, torch, ultralytics
from PIL import Image

assert ultralytics.__version__ == ULTRALYTICS_VERSION, ultralytics.__version__
torch.set_num_threads(os.cpu_count() or 4)
ENV = {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
       "cpu_count": os.cpu_count(), "device": "cpu"}
print(json.dumps(ENV, indent=2))

## 3. No-tumor images and de-duplication

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
NEG_DIR_NAMES = {"notumor", "no_tumor", "no tumor", "no"}
neg_dirs = sorted({p for p in KAGGLE_INPUT.rglob("*") if p.is_dir() and p.name.lower() in NEG_DIR_NAMES
                   and "cv5fold" not in str(p)})
assert neg_dirs, f"No 'notumor' folder found under {KAGGLE_INPUT}: add the 4-class dataset as input."
files = sorted(f for d in neg_dirs for f in d.rglob("*") if f.suffix.lower() in IMG_EXT)
print("Folders:", *[f"{d} ({sum(1 for f in files if f.parent == d)} images)" for d in neg_dirs], sep="\n  ")


def dhash(img, size=8):
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).ravel()), 2)


rows = []
for f in files:
    raw = f.read_bytes()
    with Image.open(f) as im:
        rows.append({"path": str(f), "split": f.parent.parent.name, "md5": hashlib.md5(raw).hexdigest(),
                     "dhash": dhash(im), "width": im.width, "height": im.height, "mode": im.mode})
neg = pd.DataFrame(rows)
n_all = len(neg)
# Main set: byte-identical files removed. Sensitivity subset: one image per identical 64-bit
# difference hash (re-encoded or resized copies, but possibly also very similar distinct slices).
neg = neg.drop_duplicates("md5").reset_index(drop=True)
neg["dhash_first"] = ~neg.duplicated("dhash")
neg.to_csv(WORK / "notumor_images.csv", index=False)
print(f"{n_all} images, {n_all - len(neg)} byte-identical duplicates removed -> {len(neg)} images (main set); "
      f"{int(neg['dhash_first'].sum())} after also merging identical 64-bit hashes (sensitivity subset)")
print("image modes:", neg["mode"].value_counts().to_dict())
print("sizes (most frequent):", neg.groupby(["width", "height"]).size().sort_values(ascending=False).head(5).to_dict())

## 4. Trained models of the cross-validation

In [ ]:
MODELS = {}
for config in CONFIGS:
    for k in FOLDS:
        stem = f"test{k}_seed{SEED}__"
        cands = [p for p in KAGGLE_INPUT.rglob(f"{stem}train__weights__best.pt") if f"runs-{config}-seed{SEED}" in str(p)]
        summ = [p for p in KAGGLE_INPUT.rglob(f"{stem}summary.json") if f"runs-{config}-seed{SEED}" in str(p)]
        assert len(cands) == 1 and len(summ) == 1, (config, k, cands, summ)
        s = json.loads(summ[0].read_text())
        assert s["config"] == config and s["test_fold"] == k
        MODELS[(config, k)] = {"weights": cands[0], "operating_conf": float(s["test"]["operating_conf"]),
                               "test_mAP50": s["test"]["mAP50"],
                               "sha256": hashlib.sha256(cands[0].read_bytes()).hexdigest()[:16]}
        print(f"{config:9s} fold {k}: test mAP50 {s['test']['mAP50']:.3f}, operating conf "
              f"{MODELS[(config, k)]['operating_conf']:.3f}, weights {MODELS[(config, k)]['sha256']}")

## 5. Inference (CPU) — every detection with confidence ≥ 0.001 is recorded

In [ ]:
from ultralytics import YOLO

paths = neg["path"].tolist()
det_rows, img_rows = [], []
for (config, k), m in MODELS.items():
    model = YOLO(str(m["weights"]), task="detect")
    t0 = time.time()
    for i in range(0, len(paths), CHUNK):
        chunk = paths[i:i + CHUNK]
        results = model.predict(chunk, imgsz=IMGSZ, conf=0.001, iou=0.7, max_det=300, device="cpu", verbose=False)
        for path, r in zip(chunk, results):  # results come back in the order of the chunk
            conf = r.boxes.conf.numpy() if len(r.boxes) else np.zeros(0)
            cls = r.boxes.cls.numpy().astype(int) if len(r.boxes) else np.zeros(0, int)
            img_rows.append({"config": config, "fold": k, "path": path, "n_boxes_001": len(conf),
                             "max_conf": float(conf.max()) if len(conf) else 0.0,
                             "max_cls": int(cls[conf.argmax()]) if len(conf) else -1})
            for c, s in zip(cls, conf):
                if s >= min(THRESHOLDS + [m["operating_conf"]]):
                    det_rows.append({"config": config, "fold": k, "path": path, "cls": int(c), "conf": float(s)})
    print(f"{config:9s} fold {k}: {len(paths)} images in {time.time() - t0:.0f} s")
images = pd.DataFrame(img_rows)
dets = pd.DataFrame(det_rows, columns=["config", "fold", "path", "cls", "conf"])
images.to_csv(WORK / "notumor_max_scores.csv", index=False)
dets.to_csv(WORK / "notumor_detections.csv", index=False)

## 6. False-positive rates

In [ ]:
NAMES = {0: "meningioma", 1: "glioma", 2: "pituitary_tumor"}
summary = []
SUBSETS = {"main": set(neg["path"]), "dhash_unique": set(neg.loc[neg["dhash_first"], "path"])}
for (subset, keep), ((config, k), m) in [(s, c) for s in SUBSETS.items() for c in MODELS.items()]:
    im = images[(images.config == config) & (images.fold == k) & images.path.isin(keep)]
    d = dets[(dets.config == config) & (dets.fold == k) & dets.path.isin(keep)]
    row = {"subset": subset, "config": config, "fold": k, "n_images": len(im), "operating_conf": m["operating_conf"]}
    for name, t in [(f"{t:.2f}", t) for t in THRESHOLDS] + [("op", m["operating_conf"])]:
        dd = d[d.conf >= t]
        row[f"fp_image_rate@{name}"] = dd["path"].nunique() / len(im)
        row[f"fp_boxes_per_image@{name}"] = len(dd) / len(im)
        for c, cname in NAMES.items():
            row[f"fp_boxes_{cname}@{name}"] = int((dd.cls == c).sum())
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(WORK / "notumor_fp_summary.csv", index=False)
cols = [c for c in summary.columns if c.startswith(("fp_image_rate", "fp_boxes_per_image"))]
for subset, gs in summary.groupby("subset", sort=False):
    print(f"\n===== {subset} ({gs['n_images'].iloc[0]} images)")
    print(gs[["config", "fold", "operating_conf"] + cols].round(3).to_string(index=False))
    for config, g in gs.groupby("config", sort=False):
        print(f"{config}: " + ", ".join(f"{c} {g[c].mean():.3f} ± {g[c].std(ddof=1):.3f}" for c in cols))
    if set(CONFIGS) == {"adopted", "yolov12n"}:
        a = gs[gs.config == "adopted"].set_index("fold")
        b = gs[gs.config == "yolov12n"].set_index("fold")
        for c in cols:
            d = (b[c] - a[c]).loc[FOLDS]
            print(f"yolov12n − adopted, {c}: per fold {' '.join(f'{x:+.3f}' for x in d)}, mean {d.mean():+.3f}")

## 7. Examples (highest-scoring false positives of the fold-1 models) and archive

In [ ]:
import zipfile
ex_dir = WORK / "examples"
ex_dir.mkdir(exist_ok=True)
for config in CONFIGS:
    model = YOLO(str(MODELS[(config, 1)]["weights"]), task="detect")
    top = images[(images.config == config) & (images.fold == 1)].nlargest(N_EXAMPLES, "max_conf")
    for i, p in enumerate(top["path"], 1):
        r = model.predict(p, imgsz=IMGSZ, conf=0.25, iou=0.7, device="cpu", verbose=False)[0]
        Image.fromarray(r.plot()[..., ::-1]).save(ex_dir / f"{config}_fold1_top{i}_{Path(p).stem}.jpg", quality=90)
archive = WORK / "notumor_fp_test_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["notumor_images.csv", "notumor_max_scores.csv", "notumor_detections.csv", "notumor_fp_summary.csv"]:
        zf.write(WORK / f, f)
    for f in sorted(ex_dir.glob("*.jpg")):
        zf.write(f, f"examples/{f.name}")
    zf.writestr("environment.json", json.dumps(ENV, indent=2))
    zf.writestr("models.json", json.dumps({f"{c}_fold{k}": {kk: str(v) for kk, v in m.items()}
                                           for (c, k), m in MODELS.items()}, indent=2))
print(f"{archive} ({archive.stat().st_size / 2**20:.1f} MiB)")